# Chapter 3: The Operational Layer

This notebook sets up Snowflake Postgres as the operational store for live shipment records. Every shipment event consumed from Kafka in Chapter 5 is written here in real time. The live operations Streamlit app in Chapter 7 reads from this database to show current shipment status, active disruptions and rerouted shipments.

Snowflake Postgres is a fully managed Postgres database integrated directly into the Snowflake platform. It uses a static password and standard Postgres drivers -- the connection pattern is the same as any managed Postgres service.

## 1. Install Dependencies

All versions are pinned for reproducibility. Run this cell once.

In [1]:
%pip install psycopg2-binary==2.9.12 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import psycopg2

## 3. Configuration

Connection details are read from environment variables. No credentials are hard-coded in the notebook.

In [3]:
SNOWFLAKE_PG_HOST     = os.environ["SNOWFLAKE_PG_HOST"]
SNOWFLAKE_PG_USER     = os.environ["SNOWFLAKE_PG_USER"]
SNOWFLAKE_PG_PASSWORD = os.environ["SNOWFLAKE_PG_PASSWORD"]
SNOWFLAKE_PG_DBNAME   = os.environ["SNOWFLAKE_PG_DBNAME"]

print("Configuration set.")

Configuration set.


## 4. Connect to Snowflake Postgres

Snowflake Postgres uses standard Postgres drivers. We connect with `psycopg2` using the static password from the connection string. SSL is required for all connections.

In [4]:
conn = psycopg2.connect(
    host     = SNOWFLAKE_PG_HOST,
    user     = SNOWFLAKE_PG_USER,
    password = SNOWFLAKE_PG_PASSWORD,
    dbname   = SNOWFLAKE_PG_DBNAME,
    sslmode  = "require",
    port     = 5432
)

conn.autocommit = True
cursor = conn.cursor()

cursor.execute("SELECT version()")
print("Connected to:", cursor.fetchone()[0])

Connected to: PostgreSQL 18.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 11.5.0 20240719 (Red Hat 11.5.0-14), 64-bit


## 5. Role Setup

We create a dedicated role for the supply chain application. This role has the minimum permissions needed to read and write the shipments table -- no superuser privileges are granted to the application user.

Run this section once. If the role already exists the `DO` block skips creation cleanly.

In [5]:
cursor.execute("""
    DO $$
    BEGIN
        IF NOT EXISTS (SELECT FROM pg_roles WHERE rolname = 'supply_chain_app') THEN
            CREATE ROLE supply_chain_app LOGIN PASSWORD 'change-me-in-production';
        END IF;
    END
    $$;
""")

cursor.execute("GRANT USAGE ON SCHEMA public TO supply_chain_app")

print("Role supply_chain_app ready.")

Role supply_chain_app ready.


## 6. Create the Shipments Table

The `shipments` table is the operational store for live shipment events. Every event consumed from Kafka in Chapter 5 is written here.

The `rerouted` and `alt_warehouse_id` columns support the disruption scenario in Chapter 6 -- when a warehouse goes inactive, rerouted shipments are flagged here and the alternative warehouse is recorded.

We drop and recreate the table on each run so the notebook is safe to re-run from the start during development.

In [6]:
tables = {
    "shipments": """
        CREATE TABLE shipments (
            shipment_id      TEXT PRIMARY KEY,
            supplier_id      TEXT        NOT NULL,
            warehouse_id     TEXT        NOT NULL,
            dist_center_id   TEXT        NOT NULL,
            retailer_id      TEXT        NOT NULL,
            status           TEXT        NOT NULL,
            delay_minutes    INTEGER     NOT NULL DEFAULT 0,
            rerouted         BOOLEAN     NOT NULL DEFAULT FALSE,
            alt_warehouse_id TEXT,
            recorded_at      TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """,
    "disruption_events": """
        CREATE TABLE disruption_events (
            id            SERIAL PRIMARY KEY,
            warehouse_id  TEXT        NOT NULL,
            event_type    TEXT        NOT NULL,
            recorded_at   TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """,
}

for name, ddl in tables.items():
    cursor.execute(f"DROP TABLE IF EXISTS {name}")
    cursor.execute(ddl)
    print(f"Table {name} created.")

Table shipments created.
Table disruption_events created.


## 7. Grant Table Permissions

We grant the application role full read/write access to the shipments table and set default privileges so any future tables in this schema are accessible too.

In [7]:
cursor.execute("""
    GRANT SELECT, INSERT, UPDATE, DELETE
    ON TABLE shipments TO supply_chain_app
""")

cursor.execute("""
    GRANT SELECT, INSERT, UPDATE, DELETE
    ON TABLE disruption_events TO supply_chain_app
""")

cursor.execute("""
    ALTER DEFAULT PRIVILEGES IN SCHEMA public
    GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO supply_chain_app
""")

print("Permissions granted.")

Permissions granted.


## 8. Create Indexes

We create indexes on the columns most commonly used in queries by the Kafka consumer and the Streamlit dashboard:

- `status` -- the dashboard filters by status on every refresh
- `warehouse_id` -- disruption queries filter by warehouse
- `recorded_at` -- time-window queries in the analytics layer
- `rerouted` -- the dashboard highlights rerouted shipments

In [8]:
indexes = [
    ("idx_shipments_status",      "shipments (status)"),
    ("idx_shipments_warehouse",   "shipments (warehouse_id)"),
    ("idx_shipments_recorded_at", "shipments (recorded_at DESC)"),
    ("idx_shipments_rerouted",    "shipments (rerouted)"),
]

for name, definition in indexes:
    cursor.execute(f"CREATE INDEX IF NOT EXISTS {name} ON {definition}")

print("Indexes created.")

Indexes created.


## 9. Verify

We confirm the table and indexes exist and check the column definitions match
what we expect before handing off to Chapter 5.

In [9]:
# Table columns -- shipments
cursor.execute("""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = 'shipments'
    ORDER BY ordinal_position
""")

print("Table: shipments")
print(f"  {'Column':<20} {'Type':<20} {'Nullable':<10} {'Default'}")
print(f"  {'-'*70}")
for col, dtype, nullable, default in cursor.fetchall():
    print(f"  {col:<20} {dtype:<20} {nullable:<10} {default or ''}")

# Table columns -- disruption_events
cursor.execute("""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = 'disruption_events'
    ORDER BY ordinal_position
""")

print("\nTable: disruption_events")
print(f"  {'Column':<20} {'Type':<20} {'Nullable':<10} {'Default'}")
print(f"  {'-'*70}")
for col, dtype, nullable, default in cursor.fetchall():
    print(f"  {col:<20} {dtype:<20} {nullable:<10} {default or ''}")

# Indexes
cursor.execute("""
    SELECT indexname
    FROM pg_indexes
    WHERE tablename = 'shipments'
    ORDER BY indexname
""")

print("\nIndexes (shipments):")
for row in cursor.fetchall():
    print(f"  {row[0]}")

Table: shipments
  Column               Type                 Nullable   Default
  ----------------------------------------------------------------------
  shipment_id          text                 NO         
  supplier_id          text                 NO         
  warehouse_id         text                 NO         
  dist_center_id       text                 NO         
  retailer_id          text                 NO         
  status               text                 NO         
  delay_minutes        integer              NO         0
  rerouted             boolean              NO         false
  alt_warehouse_id     text                 YES        
  recorded_at          timestamp with time zone NO         now()

Table: disruption_events
  Column               Type                 Nullable   Default
  ----------------------------------------------------------------------
  id                   integer              NO         nextval('disruption_events_id_seq'::regclass)
  warehou

## 10. Teardown

Close the connection. The schema stays in place -- the Kafka consumer in Chapter 5 connects to the same database and writes shipment events to the `shipments` table.

In [10]:
cursor.close()
conn.close()
print("Connection closed.")

Connection closed.
